## Preprocessing 

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/Users/khairul/Downloads/MachineLearning/Demand Forecast and Allocation Engine/data/processed/Masterfile.csv')

In [4]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['week'] = df['order_purchase_timestamp'].dt.to_period('W-MON').dt.start_time

In [5]:
actual_demand = df.groupby(['week', 'customer_state', 'product_category_name']).size().reset_index(name='demand_volume')

In [6]:
unique_weeks = actual_demand['week'].unique()
unique_states = actual_demand['customer_state'].unique()
unique_categories = actual_demand['product_category_name'].unique()

In [7]:
from itertools import product
grid = list(product(unique_weeks, unique_states, unique_categories))
df_grid = pd.DataFrame(grid, columns=['week', 'customer_state', 'product_category_name'])

In [8]:
df_timeseries = pd.merge(df_grid, actual_demand, 
                         on=['week', 'customer_state', 'product_category_name'], 
                         how='left')

In [9]:
df_timeseries['demand_volume'] = df_timeseries['demand_volume'].fillna(0).astype(int)
df_timeseries = df_timeseries.sort_values(by=['customer_state', 'product_category_name', 'week']).reset_index(drop=True)

print(f"Time-series shape: {df_timeseries.shape}")

Time-series shape: (181818, 4)


## Feature Engineering

In [10]:
df_timeseries = pd.DataFrame(df_timeseries)

In [11]:
df_timeseries = df_timeseries.sort_values(by=['customer_state', 'product_category_name', 'week']).reset_index(drop=True)

df_timeseries['month'] = df_timeseries['week'].dt.month
df_timeseries['week_of_year'] = df_timeseries['week'].dt.isocalendar().week.astype(int)

In [12]:
group_keys = ['customer_state', 'product_category_name']

for i in [1, 2, 3, 4]:
    
    df_timeseries[f'demand_lag_{i}wk'] = df_timeseries.groupby(group_keys)['demand_volume'].shift(i)

In [13]:
df_timeseries['rolling_mean_4wk'] = df_timeseries.groupby(group_keys)['demand_volume'].transform(
    lambda x: x.shift(1).rolling(window=4).mean()
)

df_timeseries['rolling_std_4wk'] = df_timeseries.groupby(group_keys)['demand_volume'].transform(
    lambda x: x.shift(1).rolling(window=4).std()
)

In [62]:
df_features = df_timeseries.dropna().reset_index(drop=True)

print(f"Feature Engineering shape: {df_features.shape}")

Feature Engineering shape: (173826, 12)


In [63]:
Features = pd.DataFrame(df_features)

In [64]:
df_features

,week,customer_state,product_category_name,demand_volume,month,week_of_year,demand_lag_1wk,demand_lag_2wk,demand_lag_3wk,demand_lag_4wk,rolling_mean_4wk,rolling_std_4wk
0,2017-01-03,AC,No Category,0,1,1,0.0,0.0,0.0,0.0,0.00,0.00000
1,2017-01-10,AC,No Category,0,1,2,0.0,0.0,0.0,0.0,0.00,0.00000
2,2017-01-17,AC,No Category,0,1,3,0.0,0.0,0.0,0.0,0.00,0.00000
3,2017-01-24,AC,No Category,0,1,4,0.0,0.0,0.0,0.0,0.00,0.00000
4,2017-01-31,AC,No Category,0,1,5,0.0,0.0,0.0,0.0,0.00,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...
173821,2018-07-31,TO,utilidades_domesticas,1,7,31,0.0,0.0,1.0,0.0,0.25,0.50000
173822,2018-08-07,TO,utilidades_domesticas,0,8,32,1.0,0.0,0.0,1.0,0.50,0.57735
173823,2018-08-14,TO,utilidades_domesticas,0,8,33,0.0,1.0,0.0,0.0,0.25,0.50000
173824,2018-08-21,TO,utilidades_domesticas,0,8,34,0.0,0.0,1.0,0.0,0.25,0.50000


## Train Test Split

In [51]:
from sklearn.preprocessing import OneHotEncoder
import joblib

categorical_cols = ['product_category_name', 'customer_state']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_array = encoder.fit_transform(Features[categorical_cols])

encoded_col_names = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded_array, columns=encoded_col_names, index=Features.index)

Features = Features.drop(columns=categorical_cols)
Features = pd.concat([Features, encoded_df], axis=1)

joblib.dump(encoder, 'category_encoder.joblib')

['category_encoder.joblib']

In [52]:
Features = Features.sort_values(by=['week'])

split_index = int(len(Features) * 0.6)

train_data = Features.iloc[:split_index]
test_data = Features.iloc[split_index:]

In [53]:
drop_cols = ['demand_volume', 'week']

X_train = train_data.drop(columns=drop_cols)
y_train = train_data['demand_volume']

X_test = test_data.drop(columns=drop_cols)
y_test = test_data['demand_volume']

## Train the Baseline Model

In [55]:
import xgboost as xgb
import sklearn

base_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    objective='reg:squarederror' # Standard objective for regression
)

base_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

## Evaluate

In [56]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

base_preds = base_model.predict(X_test)

base_mae = mean_absolute_error(y_test, base_preds)
base_rmse = np.sqrt(mean_squared_error(y_test, base_preds))

print(f"MAE:  {base_mae:.2f} units")
print(f"RMSE: {base_rmse:.2f} units")

MAE:  0.53 units
RMSE: 2.04 units


## Hyperparameter tuning

In [57]:
import optuna

def objective(trial):

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'objective': 'reg:squarederror',
        'random_state': 42
    }
    
    model = xgb.XGBRegressor(**params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20) 

print(f"Best Trial RMSE: {study.best_value:.2f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-05-30 15:59:54,552] A new study created in memory with name: no-name-3344f499-f011-4fec-aa32-e6bb742a5cbe
[I 2026-05-30 16:00:06,545] Trial 0 finished with value: 1.9861378686461482 and parameters: {'n_estimators': 498, 'max_depth': 4, 'learning_rate': 0.024168002563158434, 'subsample': 0.6677549553055512, 'colsample_bytree': 0.6170653745229872, 'min_child_weight': 4}. Best is trial 0 with value: 1.9861378686461482.
[I 2026-05-30 16:00:09,820] Trial 1 finished with value: 2.0890987601375275 and parameters: {'n_estimators': 107, 'max_depth': 5, 'learning_rate': 0.12543998810191434, 'subsample': 0.9053838451535945, 'colsample_bytree': 0.9066538382088946, 'min_child_weight': 2}. Best is trial 0 with value: 1.9861378686461482.
[I 2026-05-30 16:00:18,854] Trial 2 finished with value: 2.0503609362804047 and parameters: {'n_estimators': 337, 'max_depth': 7, 'learning_rate': 0.030616868702619326, 'subsample': 0.6938929891211232, 'colsample_bytree': 0.863102255174679, 'min_child_weight'

Best Trial RMSE: 1.90
Best Parameters:
  n_estimators: 140
  max_depth: 3
  learning_rate: 0.05472236500104547
  subsample: 0.6737072592813633
  colsample_bytree: 0.7906515846413494
  min_child_weight: 8


In [58]:
best_model = xgb.XGBRegressor(**study.best_params, objective='reg:squarederror', random_state=42)
best_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7906515846413494, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05472236500104547, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=8, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=140, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)